In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [5]:
from sklearn.metrics import classification_report
ths = [19, 20, 21, 22, 23, 24, 25, 26]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 19 hidden = 0


,0,1,2,3,4,5,-1
0,0,156595,0,0,0,0,45633
1,0,126232,35,0,406,500,1363
2,0,1,82267,0,0,0,37
3,0,0,3,60925,0,0,22
4,0,58,0,0,45585,1,146
5,0,5,0,0,11,103,28
-1,0,0,0,0,0,0,0


th: 19 hidden: 0 acc:0.6957607951442045 recall:0.22565124512926005 precision:0.9662072032014228 f1:0.3658586449768898
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.97      0.23      0.37    202228
           1       0.45      0.98      0.61    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      1.00      0.99     45790
           5       0.17      0.70      0.27       147

    accuracy                           0.69    519956
   macro avg       0.76      0.82      0.71    519956
weighted avg       0.85      0.69      0.66    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,150316,0,0,0,0,51912
1,0,125951,17,0,373,379,1816
2,0,1,82226,0,0,0,78
3,0,0,3,60925,0,0,22
4,0,55,0,0,45579,0,156
5,0,5,0,0,11,102,29
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.7068655809337713 recall:0.2567003580117491 precision:0.9611019569362931 f1:0.4051810600177177
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.96      0.26      0.41    202228
           1       0.46      0.98      0.62    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      1.00      0.99     45790
           5       0.21      0.69      0.32       147

    accuracy                           0.71    519956
   macro avg       0.77      0.82      0.72    519956
weighted avg       0.85      0.71      0.67    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


...

th: 24 hidden: 5 acc:0.986985437229304 recall:0.04081632653061224 precision:0.0009047044632086852 f1:0.001770172591827703
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.04      0.00       147
           0       1.00      0.99      1.00    202228
           1       1.00      0.96      0.98    128536
           2       1.00      0.98      0.99     82305
           3       1.00      1.00      1.00     60950
           4       0.99      1.00      1.00     45790

    accuracy                           0.99    519956
   macro avg       0.83      0.83      0.83    519956
weighted avg       1.00      0.99      0.99    519956

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,200850,279,0,0,0,0,1099
1,470,120384,3,1,190,0,7488
2,0,1,80519,0,0,0,1785
3,0,0,3,60375,0,0,572
4,9,41,0,0,45512,0,228
5,0,113,0,0,15,0,19
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.978267391856234 recall:0.1292517006802721 precision:0.001697792869269949 f1:0.0033515611218909863
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.13      0.00       147
           0       1.00      0.99      1.00    202228
           1       1.00      0.94      0.97    128536
           2       1.00      0.98      0.99     82305
           3       1.00      0.99      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.98    519956
   macro avg       0.83      0.84      0.82    519956
weighted avg       1.00      0.98      0.99    519956

[0, 1, 2, 3, 4, 5, -1]
th = 26 hidden = 5


,0,1,2,3,4,5,-1
0,200595,0,0,0,0,0,1633
1,469,109139,3,0,140,0,18785
2,0,1,78244,0,0,0,4060
3,0,0,3,60126,0,0,821
4,9,39,0,0,44895,0,847
5,0,113,0,0,14,0,20
-1,0,0,0,0,0,0,0


th: 26 hidden: 5 acc:0.9494707244459146 recall:0.1360544217687075 precision:0.0007643506840938623 f1:0.0015201611370805306
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.14      0.00       147
           0       1.00      0.99      0.99    202228
           1       1.00      0.85      0.92    128536
           2       1.00      0.95      0.97     82305
           3       1.00      0.99      0.99     60950
           4       1.00      0.98      0.99     45790

    accuracy                           0.95    519956
   macro avg       0.83      0.82      0.81    519956
weighted avg       1.00      0.95      0.97    519956



# Média absolute threshold

In [6]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 19: 0.5773258421078997
Média f1 absolute th 20: 0.576715233605009
Média f1 absolute th 21: 0.5778131372573505
Média f1 absolute th 22: 0.5671130957141388
Média f1 absolute th 23: 0.6307543531186959
Média f1 absolute th 24: 0.6209113076106647
Média f1 absolute th 25: 0.62522962993054
Média f1 absolute th 26: 0.607978045307273
